# Temporal Fusion Transformer (TFT) — Simple Pipeline (No Optuna)
This notebook trains and runs **Temporal Fusion Transformer** on the competition dataset:
- Train: `./dataset/train.csv`
- Test : `./dataset/TEST_*.csv` (each file: 28-day input)
- Forecast horizon: 7 days per test file
- Metric for monitoring: sMAPE (reported), training loss: MAE
- Library: `pytorch-forecasting` (TFT implementation) + `pytorch-lightning`

> Notes
> - This is a compact, no-Optuna, single split setup designed to run end-to-end.
> - It uses only date-derived covariates to comply with "no external data".
> - Negative `sales` values are clipped to 0 by rule.
> - If you already have the libraries installed, skip the install cell.


In [34]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [35]:
# === 1) Imports & global config ===
import os, math, warnings, glob
from pathlib import Path
from datetime import timedelta


import torch
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
    from lightning.pytorch.loggers import CSVLogger

    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.metrics import MAE, SMAPE

except Exception as e:
    raise RuntimeError(
        'Missing dependencies. Please install pytorch-lightning and pytorch-forecasting. '
        'Uncomment the pip cell above and re-run.'
    ) from e

SEED = 42
pl.seed_everything(SEED, workers=False)

DATA_DIR = Path('./dataset')
RESULT_DIR = Path('./result')
RESULT_DIR.mkdir(exist_ok=True, parents=True)

MAX_ENCODER_LENGTH = 28
MAX_PREDICTION_LENGTH = 7
BATCH_SIZE = 64
MAX_EPOCHS = 60

LR = 3e-4
HIDDEN_SIZE = 128
ATTN_HEADS = 8
HIDDEN_CONT_SIZE = 32
DROPOUT = 0.2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Seed set to 42


Device: cuda


In [36]:
# === 2) Metrics & helpers ===
def smape_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute symmetric MAPE in percentage."""
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1.0, denom)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

def add_date_features(df: pd.DataFrame, date_col: str = 'date') -> pd.DataFrame:
    """Add only date-derived features allowed without external data."""
    d = pd.to_datetime(df[date_col])
    df = df.copy()
    df['year'] = d.dt.year.astype(int)
    df['month'] = d.dt.month.astype(int)
    df['day'] = d.dt.day.astype(int)
    df['dow'] = d.dt.weekday.astype(int)
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7.0)
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['eom'] = (d.dt.is_month_end).astype(int)
    df['is_weekend'] = d.dt.weekday.isin([5, 6]).astype(int)
    df['dom_sin'] = np.sin(2 * np.pi * d.dt.day / 31.0)
    df['dom_cos'] = np.cos(2 * np.pi * d.dt.day / 31.0)
    return df


In [37]:
# === 3) Load train.csv & basic cleaning ===
train_path = DATA_DIR / 'train.csv'
assert train_path.exists(), f'Train file not found: {train_path}'
train_df = pd.read_csv(train_path)

if 'store_menu' not in train_df.columns:
    if 'store_menu_id' in train_df.columns:
        train_df['store_menu'] = train_df['store_menu_id']
    else:
        train_df['store_menu'] = train_df['store'].astype(str) + '_' + train_df['menu'].astype(str)

train_df['date'] = pd.to_datetime(train_df['date'])
train_df = train_df.sort_values(['store_menu', 'date']).reset_index(drop=True)
train_df['sales'] = train_df['sales'].astype(float)
train_df['sales'] = np.clip(train_df['sales'], 0.0, None)

train_df = add_date_features(train_df, 'date')
global_min_date = train_df['date'].min()
train_df['time_idx'] = (train_df['date'] - global_min_date).dt.days.astype(int)

print(train_df.head())
print('n_series:', train_df['store_menu'].nunique(), 'rows:', len(train_df))


   date_ordinal       date          store_menu       store     menu  sales  \
0        738521 2023-01-01  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트    0.0   
1        738522 2023-01-02  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트    0.0   
2        738523 2023-01-03  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트    0.0   
3        738524 2023-01-04  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트    0.0   
4        738525 2023-01-05  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트    0.0   

   year  month  day  dow   dow_sin   dow_cos  month_sin  month_cos  eom  \
0  2023      1    1    6 -0.781831  0.623490        0.5   0.866025    0   
1  2023      1    2    0  0.000000  1.000000        0.5   0.866025    0   
2  2023      1    3    1  0.781831  0.623490        0.5   0.866025    0   
3  2023      1    4    2  0.974928 -0.222521        0.5   0.866025    0   
4  2023      1    5    3  0.433884 -0.900969        0.5   0.866025    0   

   is_weekend   dom_sin   dom_cos  time_idx  
0           1  0.201299  0.979530 

In [38]:
# === 4) Build TimeSeriesDataSet for TFT ===
training_cutoff = train_df['time_idx'].max() - MAX_PREDICTION_LENGTH
print('training_cutoff (time_idx):', training_cutoff)

known_future_reals = [
    'dow_sin','dow_cos','month_sin','month_cos','eom',
    'is_weekend','dom_sin','dom_cos'   # ← add
]
target = 'sales'

training = TimeSeriesDataSet(
    train_df[lambda x: x.time_idx <= training_cutoff],
    time_idx='time_idx',
    target=target,
    group_ids=['store_menu'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=['store_menu'],
    time_varying_known_reals=['time_idx'] + known_future_reals,
    time_varying_unknown_reals=[target],
    target_normalizer=None,
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    train_df,
    predict=True,
    stop_randomization=True,
    min_prediction_idx=training_cutoff + 1,
)

num_workers = max(2, os.cpu_count() // 2)
train_loader = training.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=num_workers, pin_memory=True
)
valid_loader = validation.to_dataloader(
    train=False, batch_size=BATCH_SIZE * 2, num_workers=num_workers, pin_memory=True
)

training_cutoff (time_idx): 524


In [39]:
# 교체
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

# === 가속기/정밀도 ===
pl_accelerator = "gpu" if torch.cuda.is_available() else "cpu"
precision_val = "16-mixed" if torch.cuda.is_available() else 32  # CPU이면 16-mixed 금지

# === 로거 ===
logger = CSVLogger(save_dir=str(RESULT_DIR), name="logs")  # RESULT_DIR는 기존 변수 사용

# === 콜백들 ===
es_cb = EarlyStopping(monitor="val_loss", min_delta=0.0, patience=8, mode="min")

ckpt_dir = RESULT_DIR / "ckpt"
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_cb = ModelCheckpoint(
    dirpath=str(ckpt_dir),
    filename="tft_best",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=False,
    auto_insert_metric_name=False,
)

lr_cb = LearningRateMonitor(logging_interval="epoch")


In [ ]:
# === 5) TFT model & Trainer ===
import torch
# 텐서코어 설정(경고 제거용)
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
precision_val = "bf16-mixed" if use_bf16 else 32    # FP16(‘16-mixed’) 금지



pl_accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
logger = CSVLogger(save_dir=str(RESULT_DIR / 'logs'), name='tft_simple')

es_cb = EarlyStopping(monitor='val_loss', min_delta=0.0, patience=8, mode='min')  # 5 → 8

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    deterministic=False,
    precision=precision_val,          # ← 위에서 계산한 값 사용
    gradient_clip_val=1.0,
    accumulate_grad_batches=2,
    callbacks=[es_cb, ckpt_cb, lr_cb],
    logger=logger,
)



ckpt_cb = ModelCheckpoint(
    dirpath=str(RESULT_DIR / 'ckpt'),
    filename='tft-simple-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    save_top_k=1,
    mode='min'
)
lr_cb = LearningRateMonitor(logging_interval='epoch')


tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LR,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTN_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONT_SIZE,
    loss=MAE(),
    optimizer='adam',
    reduce_on_plateau_patience=3,
    weight_decay=1e-4,     # ← add
)

print(f'Model params: {tft.size()/1e6:.2f}M')
trainer.fit(model=tft, train_dataloaders=train_loader, val_dataloaders=valid_loader)
best_path = ckpt_cb.best_model_path
print('Best checkpoint:', best_path)


# 127m for epoch 26 (out of 60)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 5.8 K  | train
3  | prescalers                         | ModuleDict                      | 896    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 34.8 K | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 129 K  | train
6  | decoder_variable_selection         | VariableSelectionN

Model params: 1.10M
Epoch 26: 100%|██████████| 1480/1480 [04:43<00:00,  5.22it/s, v_num=2, train_loss_step=8.260, val_loss=4.200, train_loss_epoch=4.070]
Best checkpoint: 


In [41]:
# === 6) Evaluate on validation (sMAPE) ===
# Collect predictions
preds = tft.predict(valid_loader)  # [N, MAX_PREDICTION_LENGTH]

# Collect ground-truth from the dataloader
actuals_list = []
for _, y in valid_loader:
    y_true = y[0] if isinstance(y, (list, tuple)) else y   # [B, MAX_PREDICTION_LENGTH]
    actuals_list.append(y_true)
import torch
actuals = torch.cat(actuals_list, dim=0)

# Flatten and compute sMAPE with the helper defined earlier
y_pred_flat = preds.detach().cpu().numpy().reshape(-1)
y_true_flat = actuals.detach().cpu().numpy().reshape(-1)

val_smape = smape_np(y_true_flat, y_pred_flat)
print(f"Validation sMAPE: {val_smape:.3f}%")


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Validation sMAPE: 148.508%


In [42]:
# === helper: robustly extract (preds, series_ids) from model.predict(...) ===
import numpy as np, torch
from collections.abc import Mapping

def _grab_preds_index(out, horizon: int):
    """Return (preds: np.ndarray [N,horizon], series_ids: List[str]) from variable predict() outputs."""
    preds, index = None, None

    # flatten one level
    flat = (list(out) if isinstance(out, (list, tuple)) else [out])

    # 1) find index carrying 'store_menu'
    for el in flat:
        if isinstance(el, Mapping) and 'store_menu' in el:
            index = el
            break
        # pandas DataFrame case
        if hasattr(el, 'columns') and ('store_menu' in getattr(el, 'columns', [])):
            index = {'store_menu': el['store_menu'].tolist()}
            break

    # 2) find predictions shaped as [N, horizon]
    cand = []
    for el in flat:
        if torch.is_tensor(el):
            arr = el.detach().cpu().numpy()
        elif isinstance(el, np.ndarray):
            arr = el
        else:
            continue
        if arr.ndim == 2 and arr.shape[1] == horizon:
            cand.append(arr)
        elif arr.ndim == 1 and arr.size % horizon == 0:
            cand.append(arr.reshape(-1, horizon))
    if cand:
        preds = cand[0]

    if preds is None:
        raise ValueError("Could not find predictions in model.predict() output")
    if index is None:
        raise ValueError("Could not find 'store_menu' in model.predict() output")

    ids = index['store_menu']
    if hasattr(ids, 'tolist'):
        ids = ids.tolist()
    series_ids = [str(x) for x in ids]
    return np.asarray(preds), series_ids


In [43]:
# === 7) Inference helpers for TEST files ===
def build_future_frame(df_test: pd.DataFrame, min_ref_date: pd.Timestamp) -> pd.DataFrame:
    """Append 7 future rows per series, add features and time_idx based on TRAIN min date."""
    df = df_test.copy()
    df['date'] = pd.to_datetime(df['date'])

    # last date per series inside THIS test file
    last_by_id = df.groupby('store_menu')['date'].max()

    # future 7 days per series
    fut_rows = []
    for sid, last_d in last_by_id.items():
        for i in range(1, MAX_PREDICTION_LENGTH + 1):
            fut_rows.append({'store_menu': sid, 'date': last_d + timedelta(days=i), 'sales': np.nan})
    fut = pd.DataFrame(fut_rows)

    df_all = pd.concat([df, fut], ignore_index=True)

    # features + time index aligned to the TRAIN global min date
    df_all = add_date_features(df_all, 'date')
    df_all['time_idx'] = (pd.to_datetime(df_all['date']) - min_ref_date).dt.days.astype(int)

    # fill future targets to avoid NA-ratio errors (no leakage at prediction time)
    is_future = df_all['date'] > df_all['store_menu'].map(last_by_id)
    df_all.loc[is_future, 'sales'] = 0.0   # or .groupby('store_menu')['sales'].ffill().fillna(0.0)

    df_all = df_all.sort_values(['store_menu', 'date']).reset_index(drop=True)
    return df_all


def preds_to_wide(series_ids, pred_matrix, last_date):
    """Return a wide DataFrame: index=horizon dates, columns=store_menu."""
    horizon = [last_date + timedelta(days=i) for i in range(1, MAX_PREDICTION_LENGTH + 1)]
    wide = pd.DataFrame(index=horizon)
    pred_matrix = np.asarray(pred_matrix)  # shape [num_series, 7]
    for i, sid in enumerate(series_ids):
        wide[str(sid)] = pred_matrix[i, :]
    wide.index.name = 'date'
    return wide


def predict_test_file(test_csv_path: Path, model_ckpt: str | None = None) -> Path:
    """Predict next 7 days for all store_menu in one TEST_XX.csv and save wide CSV."""
    df_t = pd.read_csv(test_csv_path)
    if 'store_menu' not in df_t.columns:
        if 'store_menu_id' in df_t.columns:
            df_t['store_menu'] = df_t['store_menu_id']
        else:
            df_t['store_menu'] = df_t['store'].astype(str) + '_' + df_t['menu'].astype(str)

    df_t['date'] = pd.to_datetime(df_t['date'])
    df_t = df_t.sort_values(['store_menu', 'date']).reset_index(drop=True)
    df_t['sales'] = np.clip(df_t['sales'].astype(float), 0.0, None)

    # build inference frame
    df_pred = build_future_frame(df_t, global_min_date)

    # predict only the FINAL future window per series in this TEST file
    last_date = df_t['date'].max()                     # shared horizon anchor for this file
    last_time_idx = int((last_date - global_min_date).days)
    pred_ds = TimeSeriesDataSet.from_dataset(
        training, df_pred, predict=True, stop_randomization=True,
        min_prediction_idx=last_time_idx + 1
    )
    pred_loader = pred_ds.to_dataloader(
        train=False, batch_size=BATCH_SIZE * 4, num_workers=num_workers, pin_memory=True
    )
    model = tft if model_ckpt is None or not Path(model_ckpt).exists() \
            else TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

    # robust returns across PF versions
    out = model.predict(pred_loader, return_index=True, return_x=False)
    preds, series_ids = _grab_preds_index(out, MAX_PREDICTION_LENGTH)

    # wide output per Data description: rows = next 7 days, cols = store_menu
    wide = preds_to_wide(series_ids, preds, last_date)
    out_path = RESULT_DIR / f"tft_preds_{test_csv_path.stem}.csv"
    wide.to_csv(out_path, index=True)
    print(f"Saved: {out_path}  shape={wide.shape}")
    return out_path


In [44]:
# === 8) Batch inference for all TEST_*.csv ===
# Runs predict_test_file() on every TEST_XX.csv and saves wide CSVs under ./result

from pathlib import Path
import traceback

DATA_DIR = Path("./dataset")
RESULT_DIR = Path("./result")
RESULT_DIR.mkdir(exist_ok=True, parents=True)

# Option: use a saved checkpoint instead of the in-memory model
USE_CKPT = True
CKPT_PATH = RESULT_DIR / "tft_simple_final.ckpt"

test_files = sorted(DATA_DIR.glob("TEST_*.csv"))
print("Found test files:", [p.name for p in test_files])

saved = []
for fp in test_files:
    try:
        ckpt_arg = str(CKPT_PATH) if (USE_CKPT and CKPT_PATH.exists()) else None
        out_path = predict_test_file(fp, model_ckpt=ckpt_arg)
        saved.append(str(out_path))
    except Exception as e:
        print(f"[FAIL] {fp}: {e}")
        traceback.print_exc(limit=1)

print("Generated files:", saved)


Found test files: ['TEST_00.csv', 'TEST_01.csv', 'TEST_02.csv', 'TEST_03.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_06.csv', 'TEST_07.csv', 'TEST_08.csv', 'TEST_09.csv']


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Saved: result/tft_preds_TEST_00.csv  shape=(7, 193)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Saved: result/tft_preds_TEST_01.csv  shape=(7, 193)
Saved: result/tft_preds_TEST_02.csv  shape=(7, 193)


...

Saved: result/tft_preds_TEST_06.csv  shape=(7, 193)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Saved: result/tft_preds_TEST_07.csv  shape=(7, 193)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Saved: result/tft_preds_TEST_08.csv  shape=(7, 193)
Saved: result/tft_preds_TEST_09.csv  shape=(7, 193)
Generated files: ['result/tft_preds_TEST_00.csv', 'result/tft_preds_TEST_01.csv', 'result/tft_preds_TEST_02.csv', 'result/tft_preds_TEST_03.csv', 'result/tft_preds_TEST_04.csv', 'result/tft_preds_TEST_05.csv', 'result/tft_preds_TEST_06.csv', 'result/tft_preds_TEST_07.csv', 'result/tft_preds_TEST_08.csv', 'result/tft_preds_TEST_09.csv']


In [45]:
import re
from pathlib import Path
import pandas as pd

RESULT_DIR = Path("./result")

# 1) 예측 CSV들 로드 후 하나로 세로 결합
def num_key(p: Path):
    m = re.search(r"TEST_(\d+)", p.name)
    return int(m.group(1)) if m else 0

pred_paths = sorted(RESULT_DIR.glob("tft_preds_TEST_*.csv"), key=num_key)
dfs = [pd.read_csv(p) for p in pred_paths]
df = pd.concat(dfs, ignore_index=True)

# 2) sample_submission의 첫 열로 date 열 교체
sample = pd.read_csv(RESULT_DIR / "sample_submission.csv")
assert len(sample) == len(df), "행 수가 다르다"
df.iloc[:, 0] = sample.iloc[:, 0].values  # 첫 열 치환
df.columns = [sample.columns[0]] + df.columns.tolist()[1:]  # 열 이름도 맞춤


# 3) 저장 (UTF-8-SIG)
out_path = RESULT_DIR / "submission_tft_wonjun.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"saved -> {out_path}")


saved -> result/submission_tft_wonjun.csv


In [46]:
import pandas as pd

# 1) Load the file
df = pd.read_csv("./result/submission_tft_wonjun.csv")

# 2) Replace negative values and values between 0 and 1 with 1
df = df.applymap(lambda x: 1 if (isinstance(x, (int, float)) and (x < 0 or 0 <= x <= 1)) else x)

# 3) Save back if needed
df.to_csv("./result/submission_tft_wonjun_final.csv", index=False, encoding="utf-8-sig")

print("Done! Saved to ./result/submission_tft_wonjun_final.csv")


Done! Saved to ./result/submission_tft_wonjun_final.csv
